In [ ]:
import pandas as pd
import numpy as np
import sklearn as sk
from sklearn import metrics
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R
install.packages("countrycode")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/countrycode_1.8.0.tar.gz'
Content type 'application/x-gzip' length 539971 bytes (527 KB)
downloaded 527 KB


The downloaded source packages are in
	‘/tmp/Rtmp5BeMzj/downloaded_packages’


In [ ]:
mida = pd.read_csv("/content/MIDA 5.0.csv") ## Interstate conflict, record per conflict (1816-2014)
midb = pd.read_csv("/content/MIDB 5.0.csv") ## Interstate conflict, record per country involved in conflict (1816-2014)
nmc =  pd.read_csv("/content/NMC-60-abridged.csv") ## Capabilities of war (1816-2016)
vdem = pd.read_csv("/content/V-Dem-CY-Core-v16_filtered.csv") ## Democracy indexes (1789-2025)
orgviol = pd.read_csv("/content/organizedviolencecy_v25_1.csv") ## Organize intrastate violence (1989-2024)
cow_ccode = pd.read_csv("/content/COW-country-codes.csv") ## Correlates of war country codes
realgdp = pd.read_csv("RealGDP.csv") ## Historical economic data
natres = pd.read_csv("API_NY.GDP.TOTL.RT.ZS_DS2_en_csv_v2_83195.csv",skiprows=3) ## Natural resource rent as a proportion of GDP.

In [ ]:
# Selecting only important variables and renaming some columns for merging later
mida = mida[["dispnum", "styear", "endyear", "fatality", "fatalpre", "maxdur", "mindur", "hostlev", "recip", "numa", "numb", "ongo2014"]]
nmc = nmc[["stateabb", "ccode", "year", "cinc"]]
vdem = vdem[["country_name", "country_text_id", "country_id", "year", "v2x_polyarchy", "v2x_libdem", "v2x_partipdem", "v2x_delibdem", "v2x_egaldem"]]
vdem = vdem.rename(columns={"country_text_id" : "stateabb"})
orgviol = orgviol[["country_id_cy", "region_cy", "year_cy", "sb_exist_cy", "sb_dyad_count_cy", "sb_dyad_ids_cy", "sb_dyad_names_cy", "sb_intrastate_exist_cy", "sb_intrastate_dyad_count_cy", "sb_intrastate_dyad_ids_cy", "sb_intrastate_main_govt_inv_incomp_cy", "sb_interstate_exist_cy", "sb_interstate_dyad_count_cy", "sb_interstate_dyad_ids_cy", "sb_interstate_main_govt_inv_incomp_cy", "ns_exist_cy", "ns_dyad_count_cy", "ns_dyad_ids_cy", "os_exist_cy", "os_dyad_count_cy", "os_dyad_ids_cy", "os_main_govt_inv_cy", "os_nsgroup_inv_cy", "cumulative_total_deaths_parties_in_orgvio_cy", "cumulative_total_deaths_civilians_in_orgvio_cy", "cumulative_total_deaths_unknown_in_orgvio_cy", "cumulative_total_deaths_in_orgvio_best_cy"]]
orgviol = orgviol.rename(columns={"country_id_cy" : "ccode", "year_cy" : "year", "cumulative_total_deaths_parties_in_orgvio_cy" : "ctd_parties", "cumulative_total_deaths_civilians_in_orgvio_cy" : "ctd_civ", "cumulative_total_deaths_unknown_in_orgvio_cy" : "ctd_unk", "cumulative_total_deaths_in_orgvio_best_cy" : "ctd_best"})
cow_ccode = cow_ccode.drop_duplicates()
realgdp = realgdp.rename(columns={"Year" : "year"})
natres.drop(columns=["Country Name", "Indicator Name", "Indicator Code", "1960", "1961", "1962", "1963", "1964", "1965", "1966", "1967", "1968", "1969"], inplace=True)

In [ ]:
# We can ignore all these ambiguous matches – they either correspond to non-
# countries or are hard to categorize for historical reasons – i.e.
# Serbia/Yugoslavia.

In [ ]:
%%R -i natres -o natres
codes <- natres[['Country Code']]
library(countrycode)
natres[['stateabb']] <- countrycode(codes, origin="wb", destination="cowc")

In addition: Warning message:
Some values were not matched unambiguously: ABW, AFE, AFW, ARB, ASM, BMU, CEB, CHI, CSS, CUW, CYM, EAP, EAR, EAS, ECA, ECS, EMU, EUU, FCS, FRO, GIB, GRL, GUM, HIC, HKG, HPC, IBD, IBT, IDA, IDB, IDX, IMN, INX, LAC, LCN, LDC, LIC, LMC, LMY, LTE, MAC, MAF, MEA, MIC, MNA, MNP, NAC, NCL, OED, OSS, PRE, PRI, PSE, PSS, PST, PYF, SAS, SRB, SSA, SSF, SST, SXM, TCA, TEA, TEC, TLA, TMN, TSA, TSS, UMC, VGB, VIR, WLD
To fix unmatched values, please use the `custom_match` argument. If you think the default matching rules should be improved, please file an issue at https://github.com/vincentarelbundock/countrycode/issues
 


In [ ]:
# The original natural resource rent .csv is structured in an awkward way.
# Here, we transpose it so it can be merged directly into the other data.
natres_clean = pd.DataFrame(columns=['stateabb', 'year', 'nat_res_rent'])
for index, row in natres.iterrows():
  stateabb = row['stateabb']
  if stateabb == "None" or stateabb is None:
    continue
  for year in range(1970, 2022):
    nat_res_rent = row[str(year)]
    if nat_res_rent == np.nan or nat_res_rent == "NaN":
      continue
    natres_clean = pd.concat([natres_clean, pd.DataFrame({'stateabb': [stateabb], 'year': [year], 'nat_res_rent': [nat_res_rent]})])

/tmp/ipykernel_6915/865725233.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  natres_clean = pd.concat([natres_clean, pd.DataFrame({'stateabb': [stateabb], 'year': [year], 'nat_res_rent': [nat_res_rent]})])


In [ ]:
# Clean up the economic data. For some reason, multiple types of data are
# in the same column. This splits it off.
realgdp.dropna(inplace=True)
realgdp = realgdp.reset_index(drop=True)
realgdp['gdp'] = np.nan
realgdp['gdp_percent_change'] = np.nan

# These are all either not countries or do not work with the other datasets.
unusable = {"Africa", "Asia", "Asia and Oceania", "Asia Less Japan",
                 "Belgium Luxembourg", "BRIICs",
                 "East Asia", "East Asia Less Japan",
                 "Euro Zone", "Europe", "Europe and Central Asia",
                 "European Union 15", "European Union 27",
                 "Former Centrally Planned Economies", "Former Soviet Union",
                 "High Income Countries", "High Income Countries less USA",
                 "Latin America",
                 "Low Income Countries", "Lower-Middle Income Countries",
                 "Middle East", "Middle East and North Africa",
                 "North Africa", "North America",
                 "Oceania", "Other Asia Oceania",
                 "Other Caribbean Central America", "Other Central Europe",
                 "Other East Asia", "Other Europe", "Other Former Soviet Union",
                 "Other Middle East", "Other North Africa", "Other Oceania",
                 "Other South America", "Other South America Outlook",
                 "Other South Asia", "Other Southeast Asia",
                 "Other Southeast Asia Outlook", "Other Sub-Saharan Africa",
                 "Other West African Community", "Other Western Europe",
                 "Recently Acceded Countries",
                 "South America", "South Asia", "Southeast Asia",
                 "Sub-Saharan Africa",
                 "Upper-Middle Income Countries",
                 "USMCA",
                 "World", "World Less USA",
                 "Hong Kong", "Macau", "Puerto Rico",
                 "Serbia"}

# Remove ineligible states/regions from the dataset.
realgdp = realgdp[~realgdp['Observation'].isin(unusable)]

# Properly split the Value column based on unit type.
for index, row in realgdp.iterrows():
  if row['Unit'] == 'Real GDP USD Percent Change year to year':
    realgdp.at[index, 'gdp_percent_change'] = row['Value']
  else:
    realgdp.at[index, 'gdp'] = row['Value']
realgdp = realgdp.drop(columns=['Unit', 'Value'])
# combine rows with the same observation and year
realgdp = realgdp.groupby(['Observation', 'year']).mean().reset_index()

In [ ]:
# Removed max and min duration for MIDA conflicts and combined them
# into a mean duration variable.
mida["meandur"] = mida.apply(lambda row: (row["maxdur"] + row["mindur"])/2, axis=1)
mida = mida.drop(columns=["maxdur", "mindur"])
mida.head()

,dispnum,styear,endyear,fatality,fatalpre,hostlev,recip,numa,numb,ongo2014,meandur
0,2,1902,1903,0,0,3,0,1,1,0,193.0
1,3,1913,1913,0,0,3,0,1,1,0,177.0
2,4,1946,1946,2,-9,4,1,1,1,0,183.0
3,7,1951,1952,2,-9,4,1,1,1,0,106.0
4,8,1856,1857,6,-9,5,1,1,1,0,242.0


In [ ]:
# orgviol and nmc have same ccode so we will merge them into one dataframe
df1 = pd.merge(nmc, orgviol, on=["year", "ccode"], how="outer")

In [ ]:
# Some stateabb are still NaN even though the country code exists
# Create dictionary of ccode to stateabb and a stateabb to ccode, based on
# current dataframe pairs
ccode_stateabb = df1[["ccode", "stateabb"]].dropna().drop_duplicates()
ccmap = dict(zip(ccode_stateabb["ccode"], ccode_stateabb["stateabb"]))

# Set the state abreviation for every row based on previously-defined dictionary
def get_stateabb(row):
  ccode = row["ccode"]
  if ccode in ccmap:
    return ccmap[ccode]
  else:
    return pd.NA

df1["stateabb"] = df1.apply(get_stateabb, axis=1)

In [ ]:
# Use the COW project's defined ccode to stateabb function, to fill in rows
# with ccode but without state abreviation
def get_ccode_cow(row):
  ccode = row['ccode']
  if ccode in cow_ccode['CCode'].values:
    return cow_ccode[cow_ccode["CCode"]==ccode]['StateAbb'].tolist()[0]
  else:
    return pd.NA

df1["ccode"] = df1.apply(get_ccode_cow, axis=1)

In [ ]:
# Even after assigning ccode and stateabb based on given resources, there are
# still 108 missing data points. These 108 rows only have 3 ccodes
df1[df1['stateabb'].isna()]['ccode'].unique()
# These three small island countries dont have state abreviations, presumably
# because they do not have militaries.

array([<NA>], dtype=object)

In [ ]:
%%R -i vdem -i realgdp -o vdem -o realgdp
vdemcodes <- vdem[['country_id']]
gdp_names <- realgdp[['Observation']]

library(countrycode)
vdem[['stateabb']] <- countrycode(vdemcodes, origin="vdem", destination="cowc")
realgdp[['stateabb']] <- countrycode(gdp_names, origin="country.name.en", destination="cowc")

In addition: Warning message:
Some values were not matched unambiguously: 128, 138, 139, 167, 198, 209, 355, 358, 359, 362, 363, 364, 365, 366, 373
To fix unmatched values, please use the `custom_match` argument. If you think the default matching rules should be improved, please file an issue at https://github.com/vincentarelbundock/countrycode/issues
 


In [ ]:
# Removing vestigial columns
vdem = vdem.drop(columns=["country_id"])
realgdp = realgdp.drop(columns=["Observation"])

# Combining nmc, vdem and orgviol into one dataframe
df = pd.merge(df1, vdem, on=["year", "stateabb"], how="outer")
df = pd.merge(df, realgdp, on=["year", "stateabb"], how="outer")
df = pd.merge(df, natres_clean, on=["year", "stateabb"], how="outer")

In [ ]:
stateabbmap = dict(zip(ccode_stateabb["stateabb"], ccode_stateabb["ccode"]))

# Cleaning up some of the missing country codes
def get_ccode(row):
  stateabb = row["stateabb"]
  if stateabb in stateabbmap:
    return stateabbmap[stateabb]
  else:
    return pd.NA

df["ccode"] = df.apply(get_ccode, axis=1)

In [ ]:
# Use MIDB to get identities states involved in MIDA conflicts
# Adds country codes from side A and side B to each conflict record
midb = midb[["dispnum", "ccode","stabb", "sidea"]]

mida["acountries"] = mida.apply(lambda row: midb[((row["dispnum"] == midb["dispnum"]) & (midb["sidea"] == 1))]["ccode"].to_numpy(), axis=1)
mida["bcountries"] = mida.apply(lambda row: midb[((row["dispnum"] == midb["dispnum"]) & (midb["sidea"] == 0))]["ccode"].to_numpy(), axis=1)

In [ ]:
df[df['ccode'].isna()]['stateabb'].unique()

array([None, <NA>], dtype=object)

In [ ]:
df[~df["sb_intrastate_dyad_ids_cy"].isna()]

,stateabb,ccode,year,cinc,region_cy,sb_exist_cy,sb_dyad_count_cy,sb_dyad_ids_cy,sb_dyad_names_cy,sb_intrastate_exist_cy,...,ctd_best,country_name,v2x_polyarchy,v2x_libdem,v2x_partipdem,v2x_delibdem,v2x_egaldem,gdp,gdp_percent_change,nat_res_rent
22979,AAB,58,1989,0.000003,Americas,0.0,0.0,NO_DYAD,NO_DYAD,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.771475,5.252226,0.000000
22980,AFG,700,1989,0.001282,Asia,1.0,5.0,724; 726; 727; 729; 732,Government of Afghanistan - Hizb-i Islami-yi A...,1.0,...,5411.0,Afghanistan,0.094,0.047,0.025,0.031,0.114,4.497861,-23.554617,NaN
22981,ALB,339,1989,0.000506,Europe,0.0,0.0,NO_DYAD,NO_DYAD,0.0,...,0.0,Albania,0.173,0.058,0.050,0.049,0.226,6.224525,9.836549,8.847488
22982,ALG,615,1989,0.003645,Africa,0.0,0.0,NO_DYAD,NO_DYAD,0.0,...,0.0,Algeria,0.222,0.125,0.072,0.212,0.214,78.086399,4.400002,15.074690
22983,AND,232,1989,NaN,Europe,0.0,0.0,NO_DYAD,NO_DYAD,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30620,<NA>,<NA>,2024,NaN,Asia,0.0,0.0,NO_DYAD,NO_DYAD,0.0,...,0.0,Palestine/West Bank,0.212,0.118,0.151,0.073,0.184,NaN,NaN,NaN
30621,<NA>,<NA>,2024,NaN,Asia,0.0,0.0,NO_DYAD,NO_DYAD,0.0,...,0.0,Palestine/Gaza,0.095,0.055,0.067,0.035,0.105,NaN,NaN,NaN
30622,<NA>,<NA>,2024,NaN,Asia,0.0,0.0,NO_DYAD,NO_DYAD,0.0,...,0.0,Somaliland,0.420,0.272,0.254,0.282,0.176,NaN,NaN,NaN
30623,<NA>,<NA>,2024,NaN,Asia,0.0,0.0,NO_DYAD,NO_DYAD,0.0,...,0.0,Hong Kong,0.158,0.142,0.052,0.055,0.213,NaN,NaN,NaN


In [ ]:
# Split strings into vectors of ints and strings
def int_to_array(int_list):
  if int_list != int_list:
    return np.nan
  if int_list == "NO_DYAD":
    return [0]
  ids = int_list.split("; ")
  return [int(id) for id in ids]

df['sb_dyad_ids_cy'] = df['sb_dyad_ids_cy'].apply(int_to_array)
df['ns_dyad_ids_cy'] = df['ns_dyad_ids_cy'].apply(int_to_array)
df['sb_intrastate_dyad_ids_cy'] = df['sb_intrastate_dyad_ids_cy'].apply(int_to_array)
df['sb_interstate_dyad_ids_cy'] = df['sb_interstate_dyad_ids_cy'].apply(int_to_array)
df['os_dyad_ids_cy'] = df['os_dyad_ids_cy'].apply(int_to_array)

def str_to_array(str_list):
  if str_list != str_list:
    return np.nan
  if str_list == "NO_DYAD":
    return [""]
  ids = str_list.split("; ")
  return ids

df['sb_dyad_names_cy'] = df['sb_dyad_names_cy'].apply(str_to_array)

In [ ]:
# Export these as CSVs
df.to_csv('fow.csv', index=False) # Only fully overlapping information is from 1989-2016
mida.to_csv('mida.csv', index=False) # Includes involved countries country code and side